In [18]:
"""
FRACTURE DETECTION – MSc CS PROJECT (FULL PIPELINE + POLISH)
"""

'\nFRACTURE DETECTION – MSc CS PROJECT (FULL PIPELINE + POLISH)\n'

In [19]:
# =========================
# IMPORTS
# =========================

import os
import cv2
import json
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)
print("GPU:", len(tf.config.list_physical_devices("GPU")) > 0)

TensorFlow: 2.18.0
GPU: True


In [20]:
# =========================
# CONFIG
# =========================
CONFIG = {
    "img_size": (224, 224),
    "batch_size": 16,     # reduced for memory safety
    "epochs": 120,
    "learning_rate": 3e-4,
    "test_size": 0.2,
    "seed": 42
}

np.random.seed(CONFIG["seed"])
tf.random.set_seed(CONFIG["seed"])

In [21]:
# =========================
# DATASET PATH (VERIFIED)
# =========================
DATA_DIR = "/kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images"
assert os.path.exists(DATA_DIR), "❌ Dataset path invalid"

print("✅ Dataset found:", DATA_DIR)
print("Classes:", os.listdir(DATA_DIR))

✅ Dataset found: /kaggle/input/bone-fracture-dataset/Ultimate-Mixed-Bone-Fracture-Dataset 1/images
Classes: ['Non_fractured', 'Fractured']


In [22]:
# =========================
# PREPROCESSING (EDGE-AWARE)
# =========================
def preprocess_image(path, img_size=CONFIG["img_size"]):
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None or img.size == 0:
            return None
        if img.shape[0] < 50 or img.shape[1] < 50:
            return None

        img = cv2.resize(img, img_size, interpolation=cv2.INTER_LANCZOS4)
        if np.std(img) < 5:
            return None

        clahe = cv2.createCLAHE(2.0, (8,8))
        img = clahe.apply(img)

        edges = cv2.Sobel(img, cv2.CV_32F, 1, 1)
        edges = cv2.normalize(edges, None, 0, 255, cv2.NORM_MINMAX)

        img = 0.8 * img + 0.2 * edges
        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img, -1)
        return img
    except:
        return None

In [23]:
# =========================
# LOAD DATA
# =========================
X, y = [], []

for cls in os.listdir(DATA_DIR):
    cls_path = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_path):
        continue

    label = 1 if cls.lower().startswith("fract") else 0
    files = [f for f in os.listdir(cls_path)
             if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tif','.tiff'))]

    print(f"Loading {cls}: {len(files)} images")

    for f in files:
        img = preprocess_image(os.path.join(cls_path, f))
        if img is not None:
            X.append(img)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("\nDataset loaded")
print("Total images:", len(X))
print("Non-fractured:", np.sum(y==0))
print("Fractured:", np.sum(y==1))
print("Shape:", X.shape)

Loading Non_fractured: 8133 images


Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file
P

Loading Fractured: 7323 images


libpng warning: bKGD: invalid
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: bKGD: invalid
libpng warning: bKGD: invalid



Dataset loaded
Total images: 15456
Non-fractured: 8133
Fractured: 7323
Shape: (15456, 224, 224, 1)


In [24]:
# =========================
# TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    stratify=y,
    random_state=CONFIG["seed"]
)

In [25]:
# =========================
# CLASS WEIGHTS (FN-OPTIMIZED)
# =========================
n0 = np.sum(y_train == 0)
n1 = np.sum(y_train == 1)
total = len(y_train)

class_weight = {
    0: total / (2.0 * n0) * 0.6,
    1: total / (2.0 * n1) * 2.0
}

print("Class weights:", class_weight)

Class weights: {0: 0.5701198893329235, 1: 2.110617958347559}


In [26]:
# =========================
# DATA AUGMENTATION (SOFTER FOR 224x224)
# =========================
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.12),
])

In [27]:
# =========================
# MODEL
# =========================
def build_model():
    inputs = layers.Input((224,224,1))
    x = data_augmentation(inputs)

    for filters in [48, 96, 192]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    att = layers.Conv2D(1, 1, activation="sigmoid")(x)
    x = layers.multiply([x, att])

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs)

In [28]:
# =========================
# FOCAL LOSS
# =========================
def focal_loss(gamma=2.5, alpha=0.75):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1-y_pred)
        return -alpha * tf.pow(1-pt, gamma) * tf.math.log(pt)
    return loss

model = build_model()

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=CONFIG["learning_rate"],
        weight_decay=1e-4
    ),
    loss=focal_loss(),
    metrics=[
        "accuracy",
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 224, 224,  │          0 │ input_layer_2[0]… │
│ (Sequential)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 224, 224,  │        480 │ sequential_1[0][… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        192 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_6 (ReLU)      │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 224, 224,  │     20,784 │ re_lu_6[0][0]     │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        192 │ conv2d_8[0][0]    │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_7 (ReLU)      │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 112, 112,  │          0 │ re_lu_7[0][0]     │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 112, 112,  │          0 │ max_pooling2d_3[… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 112, 112,  │     41,568 │ dropout_4[0][0]   │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        384 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_8 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 112, 112,  │     83,040 │ re_lu_8[0][0]     │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        384 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_9 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 56, 56,    │          0 │ re_lu_9[0][0]   

 Total params: 696,466 (2.66 MB)

 Trainable params: 695,122 (2.65 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [29]:
# =========================
# CALLBACKS
# =========================
cb = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        patience=35,
        restore_best_weights=True,
        mode="max"
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=12,
        min_lr=1e-7
    )
]

In [ ]:
# =========================
# TRAIN
# =========================
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    class_weight=class_weight,
    callbacks=cb,
    verbose=1
)

Epoch 1/120


E0000 00:00:1767432008.726132      47 meta_optimizer.cc:966] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_3_1/dropout_4_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


773/773 ━━━━━━━━━━━━━━━━━━━━ 211s 261ms/step - accuracy: 0.5062 - auc: 0.6608 - loss: 0.0931 - recall: 0.9625 - val_accuracy: 0.7840 - val_auc: 0.8602 - val_loss: 0.0724 - val_recall: 0.8150 - learning_rate: 3.0000e-04
Epoch 2/120
773/773 ━━━━━━━━━━━━━━━━━━━━ 206s 266ms/step - accuracy: 0.6272 - auc: 0.8278 - loss: 0.0744 - recall: 0.9300 - val_accuracy: 0.4887 - val_auc: 0.8692 - val_loss: 0.1597 - val_recall: 0.9939 - learning_rate: 3.0000e-04
Epoch 3/120
773/773 ━━━━━━━━━━━━━━━━━━━━ 208s 269ms/step - accuracy: 0.6900 - auc: 0.8593 - loss: 0.0690 - recall: 0.9254 - val_accuracy: 0.5592 - val_auc: 0.8508 - val_loss: 0.1865 - val_recall: 0.9823 - learning_rate: 3.0000e-04
Epoch 4/120
773/773 ━━━━━━━━━━━━━━━━━━━━ 208s 269ms/step - accuracy: 0.7345 - auc: 0.8842 - loss: 0.0640 - recall: 0.9261 - val_accuracy: 0.6103 - val_auc: 0.9053 - val_loss: 0.1124 - val_recall: 0.9836 - learning_rate: 3.0000e-04
Epoch 5/120
773/773 ━━━━━━━━━━━━━━━━━━━━ 209s 270ms/step - accuracy: 0.7634 - auc: 0.903

In [31]:
# =========================
# THRESHOLD SWEEP
# =========================
y_prob = model.predict(X_test).flatten()

print("\n🔍 Threshold sweep:\n")
results = []

for t in [0.30,0.35,0.40,0.45,0.50,0.55,0.60]:
    y_pred = (y_prob > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    acc = (tp + tn) / (tp + tn + fp + fn)
    recall = tp / (tp + fn)
    spec = tn / (tn + fp)

    results.append((t, acc, recall))
    print(f"t={t:.2f} | Acc={acc*100:.2f}% | Recall={recall:.4f} | FP={fp} | FN={fn}")

97/97 ━━━━━━━━━━━━━━━━━━━━ 13s 116ms/step

🔍 Threshold sweep:

t=0.30 | Acc=88.13% | Recall=0.9911 | FP=354 | FN=13
t=0.35 | Acc=89.88% | Recall=0.9809 | FP=285 | FN=28
t=0.40 | Acc=91.69% | Recall=0.9720 | FP=216 | FN=41
t=0.45 | Acc=93.21% | Recall=0.9590 | FP=150 | FN=60
t=0.50 | Acc=95.15% | Recall=0.9461 | FP=71 | FN=79
t=0.55 | Acc=95.63% | Recall=0.9270 | FP=28 | FN=107
t=0.60 | Acc=95.25% | Recall=0.9072 | FP=11 | FN=136


In [32]:
# =========================
# BEST THRESHOLD (BALANCED)
# =========================
best_t, best_acc = 0, 0
for t, acc, recall in results:
    if acc > best_acc and recall > 0.90:
        best_acc = acc
        best_t = t

print(f"\n🎯 Best threshold: {best_t}")
print(f"Accuracy: {best_acc*100:.2f}%")


🎯 Best threshold: 0.55
Accuracy: 95.63%


In [33]:
# =========================
# FINAL EVALUATION
# =========================
y_final = (y_prob > best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_final).ravel()

print("\nFINAL CONFUSION MATRIX")
print(confusion_matrix(y_test, y_final))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, y_final, digits=4))

accuracy = (tp + tn) / (tp + tn + fp + fn)
recall = tp / (tp + fn)
specificity = tn / (tn + fp)

print("\nFINAL METRICS")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Recall (Fracture): {recall:.4f}")
print(f"Specificity: {specificity:.4f}")


FINAL CONFUSION MATRIX
[[1599   28]
 [ 107 1358]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9373    0.9828    0.9595      1627
           1     0.9798    0.9270    0.9526      1465

    accuracy                         0.9563      3092
   macro avg     0.9585    0.9549    0.9561      3092
weighted avg     0.9574    0.9563    0.9563      3092


FINAL METRICS
Accuracy: 95.63%
Recall (Fracture): 0.9270
Specificity: 0.9828


In [34]:
# =========================
# SAVE MODEL + CONFIG
# =========================
model.save("fracture_model_224_final.keras")

with open("deployment_config.json", "w") as f:
    json.dump({
        "best_threshold": best_t,
        "accuracy": float(accuracy),
        "recall": float(recall),
        "specificity": float(specificity)
    }, f, indent=2)

print("\n✅ Model and deployment config saved")


✅ Model and deployment config saved
